[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/saltfingers/saltfingers.ipynb)

# Salt Fingers

This example computes salt fingers, the double-diffusive convection of warm salty water lying over cold fresh water: water that is stably stratified, lighter over heavier, and still turns over, in narrow fingers. Warmth makes the upper layer light and salt makes it heavy, and with warmth enough the two layers rest one on the other. But heat diffuses through seawater about a hundred times faster than salt. A salty parcel pushed down loses its warmth sideways to the cold fresh water around it long before it loses its salt, comes out heavier than its neighbours and keeps sinking; a fresh parcel pushed up warms and keeps rising. Here the interface between the two layers grows fingers from a little noise, as [Radko's lectures at the Woods Hole Geophysical Fluid Dynamics Program (2008)](https://gfd.whoi.edu/wp-content/uploads/sites/18/2018/03/lecture10_136325.pdf) set out.

The algebra is the plane joined with four degenerate directions, for volume, mass, entropy and salt, `Algebra("x+y+") * Algebra("v0m0s0n0")`. A parcel of water is a point weighed by its volume, offset along the mass's, the entropy's and the salt's directions by its mass, its entropy and its salt, `volume + (density - expansion * warmth + contraction * saltiness) * mass + warmth * entropy + saltiness * salt`: a unit volume holds a little less mass the warmer it is and a little more the saltier. Salt is a second amount of the same kind as entropy, a direction with no metric. A drive is minus the gradient of a potential wedged with what it acts on, and pairs with the flux of that quantity: the temperature's drive, wedged with `entropy`, and the salt's, wedged with `salt`, are each mapped to their amount's flux, at its own rate, and push nothing. Gravity's drive, the geopotential's gradient wedged with mass, pulls on the mass alone, its commutator with `volume`, `entropy` or `salt` vanishing, so heat and salt move the water only through the mass they change. The flow carries whole parcels, and the entropy and the salt it carries out of a cell are what the planes `mv.s` and `mv.n` read off one flux, `velocity & parcels`. The momentum's own advection is left out, as the large Prandtl number of water makes it small. The two amounts differ only in how fast they spread and in the sign of the mass they add, and that difference alone makes the fingers.

In vector-calculus notation, the Oberbeck–Boussinesq equations for water holding heat and salt read $\partial_t\mathbf v = -\nabla p/\rho_0 + \nu\nabla^2\mathbf v + g(\alpha T - \beta S)\,\hat{\mathbf z}$, $\nabla\cdot\mathbf v = 0$, $\partial_t T + \mathbf v\cdot\nabla T = \kappa_T\nabla^2 T$ and $\partial_t S + \mathbf v\cdot\nabla S = \kappa_S\nabla^2 S$, here without the term $(\mathbf v\cdot\nabla)\mathbf v$; the density ratio of evenly stratified water is $R = \alpha\,\partial_z T/(\beta\,\partial_z S)$, its fingers of wavenumber $k$ are marginal where $g\,(\beta\,\partial_z S/\kappa_S - \alpha\,\partial_z T/\kappa_T) = \nu k^4$, and they grow where $1 < R < \kappa_T/\kappa_S$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Callable, Iterator
from itertools import islice

import numpy as np

from numga import Algebra, NumpyContext, stack
from numga.sparse import SparseExtensor, spdiag
from examples.util.animation import show_states
from examples.fluids.saltfingers import render, scenarios
from examples.fluids.saltfingers.core import tank
from examples.util.topology import Topology

# The plane, and the volume, the mass, the entropy and the salt a parcel holds: each a direction of its own,
# degenerate, since nothing measures a volume, a mass, an entropy or an amount of salt against a length.
ga = Algebra("x+y+") * Algebra("v0m0s0n0")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Point = ga.gatype.antivector()                                                # Antivector-
Plane = ga.gatype.vector()                                                    # Vector+
# The planes reading a place's run along the tank and up it.
Upright = ga.gatype.from_blades("x y")                                        # Vector+
Drive = ga.gatype.from_blades("xv yv xm ym xs ys xn yn")                      # Bivector+
Flux = ga.gatype.antibivector()                                               # Antibivector-
# The tank's two directions, along its length and up its depth, and the planes reading a direction's run along each.
length = mv.x.dual()                                                          # [] Point-
height = mv.y.dual()                                                          # [] Point-
directions = stack((length, height))                                          # [2] Point-
readings = stack((mv.x, mv.y))                                                # [2] Plane+
# A unit of volume, the weight of a place, a unit of mass, a unit of entropy and a unit of salt.
volume = mv.v.dual()                                                          # [] Point-
mass = mv.m.dual()                                                            # [] Point-
entropy = mv.s.dual()                                                         # [] Point-
salt = mv.n.dual()                                                            # [] Point-

## The tank

The tank is a strip of triangles one depth high, closed on itself along its length so that it has a floor and a ceiling but no ends; its geometry is read on the same strip opened at its seam, its last column the first again a period further along, and the two share their triangles, in the same order. The water starts at rest in two layers, warm and salty over cold and fresh, meeting at a thin interface in the middle whose salinity holds a little noise. Lengths are in depths, times in the time heat takes to conduct across the tank, and temperature and salinity in their differences between the two layers.

In [ ]:
# The tank, twice as long as it is deep, closed and opened at its seam, the maps reading a field on each at the other's
# vertices, and the open tank's vertices, places of unit volume.
closed, opened, closing, opening, places = tank(scenarios.n_along, scenarios.n_across, scenarios.period)
vertices = (volume + mv(Upright, places).dual()).field()                      # Point-[opened vertices]
# The two layers at rest, a half above the middle's temperature and salinity at the top and below at the bottom.
temperature, salinity = scenarios.layered(closing * (mv.y & vertices), scenarios.disturbance)   # Scalar+[vertices], Scalar+[vertices]
render.draw_tank(opened, vertices, opening, scenarios.period, temperature, salinity);

The tank's temperature, above, red where warm and blue where cold, and its salinity, below, brown where salty and green where fresh: two layers meeting at a thin interface in the middle.

## Fingering

The water's flow is a stream function, whose rise at a corner carries the water along the facing side. Gravity's drive does work on each flow the stream function can make by pulling on the mass the flow carries, `pull & (flows & parcels)`, and since a parcel's mass falls with its warmth and rises with its salt, that one pairing is all the buoyancy of both. The water's mass resists the flow's change and its viscosity the flow's turning. One flux of whole parcels carries their entropy and their salt alike, and each then spreads by its own drive, the heat ten times as fast as the salt, where in seawater it is a hundred, so that the mesh resolves the salt's edges. The density ratio is two: the warmth on top holds up twice the mass its salt adds, so the layers rest stably one on the other until the fingers grow. The frames start just before the fingers show. The cells below take about fifteen seconds, most of it stepping the water.

In [ ]:
# The water: its density, its viscosity, the Prandtl number in these units, the mass a unit volume loses per degree
# and gains per unit of salinity, salt's diffusivity relative to heat's, and gravity, which with them make the heat's
# Rayleigh number and the density ratio.
density, viscosity, expansion = scenarios.density, scenarios.prandtl, scenarios.expansion
contraction = np.array(expansion / scenarios.ratio)
salt_diffusivity = scenarios.salt_diffusivity
gravity = scenarios.rayleigh * viscosity / expansion
# The step, the steps run, the steps between frames, and the frames while the interface is still flat, not drawn.
dt, n_steps, n_stride, n_flat = scenarios.dt, scenarios.n_steps, scenarios.n_stride, scenarios.n_flat

In [ ]:
# The tank's geometry: its triangles read on the open tank and gathered onto the closed one's vertices.
sides = opened.boundary[2, 1] * (opened.d[0] * vertices)                      # [corners] Point-[faces]
# Twice a triangle's area: two of its sides joined, at unit volume, mass, entropy and salt.
double_areas = ((((sides[0] & sides[1]) & volume) & mass) & entropy) & salt   # Scalar-[faces]
areas = double_areas.norm() / 2                                               # Scalar+[faces]
# A corner's hat gradient: the plane through the facing side, along volume, mass, entropy and salt, over twice
# the area, so that it reads one from the side to the corner.
gradients = (volume & mass & entropy & salt & sides) / double_areas           # [corners] Plane+[faces]
corners = closed.incidence[2, 0]                                              # [corners] Scalar+[faces <- vertices]
# Each vertex's share of the area around it, and each triangle's mean over its corners.
cells = (~corners * (areas / 3)).sum(axis=0)                                  # Scalar+[vertices]
corner_mean = corners.mean(axis=0)                                            # Scalar+[faces <- vertices]
# A stream function rising by one at a corner carries the water along the facing side, half the side's worth
# over the triangle: each corner's flow, and its velocity.
flows = sides / 2                                                             # [corners] Point-[faces]
velocities = flows / areas                                                    # [corners] Point-[faces]
# A stream function's velocity on each triangle, its corners' velocities weighed by its values there, and what
# a flux carries out of each corner's cell: its hat gradient wedged with the flux, gathered onto the vertex.
stream_velocity = (spdiag(velocities * Scalar) * corners).sum(axis=0)         # (Point- <- Scalar+)[faces <- vertices]
cell_outflow = (~corners * spdiag(areas * (gradients ^ Flux))).sum(axis=0)    # (Point- <- Flux-)[vertices <- faces]

In [ ]:
# The water's physics on the tank's triangles: its forms, and gravity's drive.
# The kinetic form: the water's inertia reads a velocity's run along and up into its momentum, paired with the
# velocity of each other corner over the triangle, and gathered onto the vertices.
inertia = density * (readings * (readings & Point)).sum(axis=0)               # [] Plane+ <- Point-
local = areas * (inertia(velocities[:, None]) & velocities[None, :])          # [corners, corners] Scalar+[faces]
kinetic = (~corners[:, None] * spdiag(local) * corners[None, :]).sum(axis=0).sum(axis=0)   # Scalar+[vertices <- vertices]
# The vorticity at each vertex, the kinetic form's amount over its share of the area and the density, and the
# viscous form, its square over the tank.
vorticity = spdiag(1 / (density * cells)) * kinetic                           # Scalar+[vertices <- vertices]
viscous = viscosity * ~vorticity * spdiag(cells) * vorticity                  # Scalar+[vertices <- vertices]
# The temperature's drive at each corner's hat, minus its gradient wedged with the entropy it pushes, and the
# salt's, wedged with salt; the conductivity and the salt's diffusivity, each drive read across the tank's two
# directions driving its amount along them.
heat_drives, salt_drives = -(gradients ^ mv.s), -(gradients ^ mv.n)           # [corners] Drive+[faces], [corners] Drive+[faces]
conductivity = ((directions & entropy) * ((directions & entropy) & Drive)).sum(axis=0)   # [] Flux- <- Drive+
diffusivity = salt_diffusivity * ((directions & salt) * ((directions & salt) & Drive)).sum(axis=0)   # [] Flux- <- Drive+
# What a flux carries out of each corner's cell: its hat gradient wedged with the flux, read for its entropy or
# its salt, gathered onto the vertices.
local = areas * (mv.s & (gradients[:, None] ^ conductivity(heat_drives[None, :])))   # [corners, corners] Scalar+[faces]
conduction = (~corners[:, None] * spdiag(local) * corners[None, :]).sum(axis=0).sum(axis=0)   # Scalar+[vertices <- vertices]
local = areas * (mv.n & (gradients[:, None] ^ diffusivity(salt_drives[None, :])))   # [corners, corners] Scalar+[faces]
diffusion = (~corners[:, None] * spdiag(local) * corners[None, :]).sum(axis=0).sum(axis=0)   # Scalar+[vertices <- vertices]
# Gravity's drive, minus the gradient of its potential wedged with mass.
pull = -(gravity * mv.ym)                                                     # [] Drive+

In [ ]:
def holding(form: SparseExtensor, interior: SparseExtensor) -> Callable[[Scalar, Scalar], Scalar]:
    """The step of a map on a mesh's vertices solved with its walls held: given the right-hand side and the walls'
    values `[...] Scalar+[vertices]`, the values the map takes to it at the free vertices, read by `interior`, and the
    walls' values elsewhere. The map on the free vertices is built once, so that every step only substitutes."""
    free = (interior * form * ~interior) * Scalar                             # (Scalar+ <- Scalar+)[free <- free]
    return lambda rhs, walls: walls + ~interior * free.solve(interior * (rhs - form * walls))


# The free vertices, and each step's implicit map, solved with the walls held: the flow's inertia with its viscosity,
# the cells' heat with its conduction, and their salt with its slower spreading.
interior = closed.reading(closed.interior[0], 0)                              # Scalar+[free <- vertices]
flow_step = holding(kinetic / dt + viscous, interior)
heat_step = holding(spdiag(cells / dt) + conduction, interior)
salt_step = holding(spdiag(cells / dt) + diffusion, interior)
# The temperature and the salinity the walls hold.
wall_temperature = temperature - ~interior * (interior * temperature)         # Scalar+[vertices]
wall_salinity = salinity - ~interior * (interior * salinity)                  # Scalar+[vertices]

In [ ]:
def fingering(temperature: Scalar, salinity: Scalar) -> Iterator[tuple[Scalar, Scalar, Scalar]]:
    """The tank's stream function, temperature and salinity `Scalar+[vertices]` every `n_stride` steps, from rest and
    this temperature and salinity: each step gravity's drive does work on the flows the stream function can make, the
    water's mass and viscosity resist them, and the flow carries the entropy and the salt along while each one's drive
    spreads it at its own rate."""
    stream = 0 * temperature                                                  # Scalar+[vertices]
    for step in range(n_steps + 1):
        if step % n_stride == 0:
            yield stream, temperature, salinity
        # Each triangle's parcels per volume: unit volume, holding less mass the warmer and more the saltier they are,
        # and their entropy and salt.
        warmth = corner_mean * temperature                                    # Scalar+[faces]
        saltiness = corner_mean * salinity                                    # Scalar+[faces]
        parcels = (volume + (density - expansion * warmth + contraction * saltiness) * mass + warmth * entropy
                   + saltiness * salt)                                        # Point-[faces]
        # Gravity's drive does work on each corner's flow by pulling on the mass it carries.
        load = (~corners * (pull & (flows & parcels[..., None]))).sum(axis=-1)   # Scalar+[vertices]
        stream = flow_step(kinetic / dt * stream + load, 0 * stream)
        # The flow carries the parcels along: what their flux carries out of each corner's cell leaves it, its entropy
        # and its salt alike, and each one's drive spreads the rest, implicitly.
        velocity = stream_velocity(stream)                                    # Point-[faces]
        flux = velocity & parcels                                             # Flux-[faces]
        outflow = cell_outflow(flux)                                          # Point-[vertices]
        temperature = heat_step(cells / dt * temperature - (mv.s & outflow), wall_temperature)
        salinity = salt_step(cells / dt * salinity - (mv.n & outflow), wall_salinity)

In [ ]:
# Drawn as each frame comes, from just before the fingers show; the states drawn come back.
states = show_states(islice(fingering(temperature, salinity), n_flat, None), render.animate, opened, vertices, opening,
                     scenarios.period, name="saltfingers", duration_ms=scenarios.frame_duration_ms,
                     n_frames=n_steps // n_stride + 1 - n_flat)

The tank's temperature, above, and its salinity, below, from just before the fingers show. The interface's ripples grow into fingers of salty water sinking and fresh water rising, about ten along the tank, about as wide as heat crosses in the time they take to grow. In the temperature the fingers are blurred, each having traded most of its warmth with its neighbours sideways; in the salinity they keep their sharp edges and nearly all their salt, which is what keeps them moving. Their heads curl into small pairs of vortices, the shape heat–salt fingers take in two dimensions.

In [ ]:
# checks
# Gravity pulls a parcel's mass down and passes over its volume, its entropy and its salt.
pull_readings = [(mv.y & (-mv.ym).commutator(part)).to_array() for part in (mass, volume, entropy, salt)]
np.testing.assert_allclose(np.ravel(pull_readings), [-1.0, 0.0, 0.0, 0.0], atol=1e-12)
# Across the tank's middle the fingers keep their salt and trade their warmth sideways: the salinity there spans nearly
# the whole difference between the layers, and varies more than twice as much as the temperature.
middle = np.isclose((mv.y & vertices).to_array(), 0.5)
_, last_temperature, last_salinity = states[-1]
salinities = (opening * last_salinity).to_array()[middle]
temperatures = (opening * last_temperature).to_array()[middle]
assert np.ptp(salinities) > 0.9
assert temperatures.std() < salinities.std() / 2